<a href="https://colab.research.google.com/github/riccardogs/PyTorch_tutorial/blob/main/Back_propagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

When training neural networks, the most frequently used algorithm is back propagation. In this algorithm, parameters (model weights) are adjusted according to the gradient of the loss function with respect to the given parameter.

To compute those gradients, PyTorch has a built-in differentiation engine called torch.autograd. It supports automatic computation of gradient for any computational graph.

Consider the simplest one-layer neural network, with input x, parameters w and b, and some loss function. It can be defined in PyTorch in the following manner:

In [1]:
import torch


x = torch.ones(5)  # input tensor
# x = torch.ones(5): crea un tensor di input con 5 elementi tutti uguali a 1
# - Forma: (5,)
# - Valori: [1., 1., 1., 1., 1.]
# - Rappresenta un campione con 5 feature (caratteristiche)

y = torch.zeros(3)  # expected output

w = torch.randn(5, 3, requires_grad=True)
# w = torch.randn(5, 3, requires_grad=True): crea una matrice di pesi 5×3
# - Forma: (5, 3) - 5 righe (feature di input), 3 colonne (neuroni di output)
# - torch.randn: valori casuali da distribuzione normale (media 0, varianza 1)
# - requires_grad=True: dice a PyTorch di tracciare le operazioni su questo tensor
#   * Necessario per calcolare i gradienti durante la backpropagation
#   * w è un parametro che vogliamo addestrare

b = torch.randn(3, requires_grad=True)
# b = torch.randn(3, requires_grad=True): crea un vettore di bias con 3 elementi
# - Forma: (3,) - un bias per ogni neurone di output
# - Valori casuali da distribuzione normale
# - requires_grad=True: anche il bias è un parametro addestrabile

z = torch.matmul(x, w) + b
# z = torch.matmul(x, w) + b: calcola l'output del layer lineare
# - torch.matmul(x, w): moltiplicazione matriciale tra x e w
#   * x: (5,) - viene trattato come (1, 5) per broadcasting
#   * w: (5, 3)
#   * risultato: (1, 3) → (3,) dopo broadcasting
# - + b: aggiunge il bias (3,)
# - z è l'output del layer (logits), forma (3,)

loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)
# loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y): calcola la loss
# - binary_cross_entropy_with_logits: combina sigmoide + BCE in un'unica funzione
#   * Più stabile numericamente che fare sigmoide e poi BCE separatamente
# - z: logits (output del modello prima della sigmoide)
# - y: target (valori attesi tra 0 e 1)
# - Calcola l'errore tra predizione e target
# - Restituisce un tensor scalare (un numero)



In this network, w and b are parameters, which we need to optimize. Thus, we need to be able to compute the gradients of loss function with respect to those variables. In order to do that, we set the requires_grad property of those tensors.

Note: You can set the value of requires_grad when creating a tensor, or later by using x.requires_grad_(True) method.

A function that we apply to tensors to construct computational graph is in fact an object of class Function. This object knows how to compute the function in the forward direction, and also how to compute its derivative during the backward propagation step. A reference to the backward propagation function is stored in grad_fn property of a tensor. You can find more information of Function in the documentation.

In [2]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x7af9ca272a70>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x7af9e6f72c50>


# Computing Gradients

To optimize weights of parameters in the neural network, we need to compute the derivatives of our loss function with respect to parameters, namely, we need ∂loss/∂w and ∂loss/∂b under some fixed values of x and y. To compute those derivatives, we call loss.backward(), and then retrieve the values from w.grad and b.grad:

In [ ]:
loss.backward()
# loss.backward(): calcola i gradienti per tutti i tensor che hanno requires_grad=True
# - Questa è la BACKPROPAGATION (passaggio all'indietro)
# - A partire da loss, risale il grafo computazionale all'indietro
# - Calcola la derivata parziale di loss rispetto a OGNI parametro
# - I gradienti vengono ACCUMULATI negli attributi .grad dei tensor

# COSA SUCCEDE INTERNAMENTE:
# 1. Partendo da loss, segue i collegamenti .grad_fn:
#    loss ← BinaryCrossEntropyWithLogitsBackward ← z ← AddBackward ←
#    ← MatMulBackward ← w, e anche il ramo di b
#
# 2. Per ogni parametro (w e b), calcola ∂loss/∂param
#
# 3. Questi valori vengono salvati in:
#    - w.grad (stessa forma di w: 5×3)
#    - b.grad (stessa forma di b: 3)


print(w.grad)
# print(w.grad): stampa i gradienti calcolati per i pesi w
# - w.grad è un tensor della stessa forma di w: (5, 3)
# - Ogni elemento è ∂loss/∂w[i,j]
# - Valori positivi: aumentare w[i,j] aumenterebbe loss (quindi va diminuito)
# - Valori negativi: aumentare w[i,j] diminuirebbe loss (quindi va aumentato)
# - Valori grandi: quel peso ha un grande impatto sulla loss
# - Dopo l'ottimizzatore, useremo questi gradienti per aggiornare w

print(b.grad)
# print(b.grad): stampa i gradienti calcolati per i bias b
# - b.grad è un tensor della stessa forma di b: (3,)
# - Ogni elemento è ∂loss/∂b[i]
# - Indica come modificare ogni bias per ridurre la loss

# ESEMPIO CONCRETO DI OUTPUT:
# w.grad:
# tensor([[ 0.0123, -0.0456,  0.0789],
#         [ 0.0123, -0.0456,  0.0789],
#         [ 0.0123, -0.0456,  0.0789],
#         [ 0.0123, -0.0456,  0.0789],
#         [ 0.0123, -0.0456,  0.0789]])
#
# b.grad:
# tensor([-0.1234,  0.0567, -0.0891])

# NOTA IMPORTANTE:
# - .grad accumula i gradienti! Se chiami loss.backward() più volte,
#   i gradienti si SOMMANO. Per azzerarli si usa:
#   optimizer.zero_grad()  o  w.grad.zero_()

# COSA FARE CON QUESTI GRADIENTI:
# with torch.no_grad():  # disabilita il tracciamento per l'aggiornamento
#     w -= learning_rate * w.grad  # aggiorna pesi nella direzione opposta al gradiente
#     b -= learning_rate * b.grad  # per diminuire la loss
#
# Oppure meglio: usare un ottimizzatore
# optimizer = torch.optim.SGD([w, b], lr=0.01)
# optimizer.step()  # applica gli aggiornamenti usando i gradienti

# SIGNIFICATO DEI SEGNI:
# - Se w.grad è positivo: loss aumenta se w aumenta → dobbiamo DIMINUIRE w
# - Se w.grad è negativo: loss diminuisce se w aumenta → dobbiamo AUMENTARE w
# - L'aggiornamento è: w_new = w - lr * w.grad

# Disabling Gradient Tracking

By default, all tensors with requires_grad=True are tracking their computational history and support gradient computation. However, there are some cases when we do not need to do that, for example, when we have trained the model and just want to apply it to some input data, i.e. we only want to do forward computations through the network. We can stop tracking computations by surrounding our computation code with torch.no_grad() block:

In [5]:
z = torch.matmul(x, w) + b
# z = torch.matmul(x, w) + b: calcola l'output come prima
# - x: tensor di input (non richiede gradienti)
# - w: pesi con requires_grad=True
# - b: bias con requires_grad=True
# - L'operazione coinvolge tensor con requires_grad=True


print(z.requires_grad)
# print(z.requires_grad): stampa se z richiede il calcolo dei gradienti
# - Output: True
# - Perché? Se almeno un input dell'operazione richiede gradienti,
#   anche l'output richiede gradienti (per propagare all'indietro)
# - z "eredita" requires_grad=True da w e b



with torch.no_grad():
    # with torch.no_grad(): crea un contesto in cui il calcolo dei gradienti è DISATTIVATO
    # - All'interno di questo blocco, nessun tensor traccia le operazioni
    # - Utile per:
    #   * Valutazione del modello (inference)
    #   * Calcolo di metriche
    #   * Operazioni che non devono influenzare i gradienti
    #   * Risparmiare memoria (non costruisce il grafo computazionale)

    z = torch.matmul(x, w) + b
    # Stessa operazione, ma dentro il contesto no_grad
    # - Anche se w e b hanno requires_grad=True
    # - Il contesto no_grad SOPPRIME il tracciamento

print(z.requires_grad)
# print(z.requires_grad): stampa se z richiede gradienti DOPO il contesto
# - Output: False
# - Dentro no_grad, l'operazione non traccia gradienti
# - Quindi z NON richiede gradienti, anche se w e b li richiedono

# DIFFERENZA CHIAVE:

#┌────────────────────────┬─────────────────────────┬─────────────────────────┐
#│                        │ Fuori da no_grad        │ Dentro no_grad          │
#├────────────────────────┼─────────────────────────┼─────────────────────────┤
#│ Costruzione grafo      │ Sì, costruisce grafo    │ No, nessun grafo        │
#│ requires_grad di z     │ True (eredita)          │ False                   │
#│ Uso                    │ Training                │ Inference, valutazione  │
#│ Memoria                │ Alta (salva grafo)      │ Bassa (nessun grafo)    │
#│ backward() possibile   │ Sì                      │ No                      │
#└────────────────────────┴─────────────────────────┴─────────────────────────┘


# PERCHÉ USARE torch.no_grad():
# 1. Durante la VALUTAZIONE (test/validation):
#    - Non vogliamo aggiornare i pesi
#    - Non serve calcolare gradienti
#    - Risparmia memoria e velocizza

# 2. Quando calcoli metriche (accuracy, precisione, ecc.):
#    - Operazioni che non devono influenzare l'addestramento

# 3. Quando fai operazioni che non devono essere tracciate:
#    - Modifiche manuali ai pesi
#    - Operazioni di debug

# ESEMPIO TIPICO:
# Training mode (con gradienti)

# for epoch in range(num_epochs):
#    for batch in train_dataloader:
#        # Qui vogliamo gradienti
#        output = model(batch)
#        loss = loss_fn(output, target)
#        loss.backward()
#        optimizer.step()

# Validation mode (senza gradienti)
# with torch.no_grad():
#    for batch in val_dataloader:
#        # Qui NON vogliamo gradienti
#        output = model(batch)
#        accuracy = compute_accuracy(output, target)
#        # Non chiamiamo backward() qui!

True
False


Another way to achieve the same result is to use the detach() method on the tensor:

In [ ]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

There are reasons you might want to disable gradient tracking:
- To mark some parameters in your neural network as frozen parameters.

- To speed up computations when you are only doing forward pass, because computations on tensors that do not track gradients would be more efficient.

# More on Computational Graphs

Conceptually, autograd keeps a record of data (tensors) and all executed operations (along with the resulting new tensors) in a directed acyclic graph (DAG) consisting of Function objects. In this DAG, leaves are the input tensors, roots are the output tensors. By tracing this graph from roots to leaves, you can automatically compute the gradients using the chain rule.

In a forward pass, autograd does two things simultaneously:

- run the requested operation to compute a resulting tensor
- maintain the operation’s gradient function in the DAG.

The backward pass kicks off when .backward() is called on the DAG root. autograd then:

- computes the gradients from each .grad_fn,
- accumulates them in the respective tensor’s .grad attribute
- using the chain rule, propagates all the way to the leaf tensors


NOTE: DAGs are dynamic in PyTorch An important thing to note is that the graph is recreated from scratch; after each .backward() call, autograd starts populating a new graph. This is exactly what allows you to use control flow statements in your model; you can change the shape, size and operations at every iteration if needed.

# Optional Reading: Tensor Gradients and Jacobian Products

In many cases, we have a scalar loss function, and we need to compute the gradient with respect to some parameters. However, there are cases when the output function is an arbitrary tensor. In this case, PyTorch allows you to compute so-called Jacobian product, and not the actual gradient.

Instead of computing the Jacobian matrix itself, PyTorch allows you to compute Jacobian Product, it should be the same as the size of the original tensor, with respect to which we want to compute the product

In [ ]:
inp = torch.eye(4, 5, requires_grad=True)
# inp = torch.eye(4, 5, requires_grad=True): crea una matrice identità 4×5
# - torch.eye(n, m): crea una matrice con 1 sulla diagonale principale, 0 altrove
# - (4, 5): 4 righe, 5 colonne
# - requires_grad=True: vogliamo tracciare gradienti per questo tensor
# - inp sarà:
#   [[1., 0., 0., 0., 0.],
#    [0., 1., 0., 0., 0.],
#    [0., 0., 1., 0., 0.],
#    [0., 0., 0., 1., 0.]]

out = (inp+1).pow(2).t()
# out = (inp+1).pow(2).t(): una serie di operazioni
# - (inp+1): aggiunge 1 a ogni elemento → matrice con 2 sulla diagonale, 1 altrove
# - .pow(2): eleva al quadrato ogni elemento
# - .t(): trasposta la matrice (inverte righe e colonne)
# - out ha forma (5, 4) (perché è la trasposta di 4×5)

out.backward(torch.ones_like(out), retain_graph=True)
# out.backward(torch.ones_like(out), retain_graph=True): primo backward
# - torch.ones_like(out): crea un tensor di 1 della stessa forma di out
#   * Questo è il "gradiente iniziale" (gradient argument)
#   * Di solito per uno scalare si usa 1, qui out non è scalare
#   * Specifica che vogliamo gradienti uguali per tutti gli elementi di out
# - retain_graph=True: DOPO il backward, MANTIENE il grafo computazionale
#   * Normalmente, dopo backward il grafo viene cancellato per liberare memoria
#   * retain_graph=True lo mantiene per poter fare un altro backward

print(f"First call\n{inp.grad}")
# print(f"First call\n{inp.grad}"): stampa i gradienti dopo il primo backward
# - inp.grad contiene ∂out/∂inp per ogni elemento
# - I gradienti si sono ACCUMULATI in inp.grad

out.backward(torch.ones_like(out), retain_graph=True)
# out.backward(...): secondo backward sulla stessa out
# - retain_graph=True ancora: mantiene il grafo per poterlo riutilizzare

print(f"\nSecond call\n{inp.grad}")
# print(f"\nSecond call\n{inp.grad}"): stampa i gradienti dopo il secondo backward
# - NOTA: i gradienti sono DOPPI rispetto alla prima volta!
# - Perché? .backward() ACCUMULA i gradienti, non li sostituisce
# - Ogni chiamata AGGIUNGE i nuovi gradienti a quelli esistenti

inp.grad.zero_()
# inp.grad.zero_(): azzera (mette a zero) tutti i gradienti in inp.grad
# - _ indica operazione in-place (modifica direttamente inp.grad)
# - inp.grad ora è tutto zero

out.backward(torch.ones_like(out), retain_graph=True)
# out.backward(...): terzo backward, dopo aver azzerato i gradienti

print(f"\nCall after zeroing gradients\n{inp.grad}")
# print(f"\nCall after zeroing gradients\n{inp.grad}"): stampa i gradienti
# - Ora sono tornati uguali alla prima chiamata
# - Perché abbiamo azzerato prima di chiamare backward

# OUTPUT TIPICO:
# First call
# tensor([[2., 2., 2., 2., 2.],
#         [2., 2., 2., 2., 2.],
#         [2., 2., 2., 2., 2.],
#         [2., 2., 2., 2., 2.]])
#
# Second call
# tensor([[4., 4., 4., 4., 4.],
#         [4., 4., 4., 4., 4.],
#         [4., 4., 4., 4., 4.],
#         [4., 4., 4., 4., 4.]])  # DOPPIO!
#
# Call after zeroing gradients
# tensor([[2., 2., 2., 2., 2.],
#         [2., 2., 2., 2., 2.],
#         [2., 2., 2., 2., 2.],
#         [2., 2., 2., 2., 2.]])  # TORNATO ALL'ORIGINALE

# PUNTI CHIAVE:
# 1. I gradienti si ACCUMULANO (sum) a ogni .backward()
# 2. retain_graph=True mantiene il grafo per backward multipli
# 3. .zero_() azzera i gradienti accumulati
# 4. Il parametro gradient in .backward() specifica i pesi per i gradienti

Notice that when we call backward for the second time with the same argument, the value of the gradient is different. This happens because when doing backward propagation, PyTorch accumulates the gradients, i.e. the value of computed gradients is added to the grad property of all leaf nodes of computational graph. If you want to compute the proper gradients, you need to zero out the grad property before. In real-life training an optimizer helps us to do this.

Note
Previously we were calling backward() function without parameters. This is essentially equivalent to calling backward(torch.tensor(1.0)), which is a useful way to compute the gradients in case of a scalar-valued function, such as loss during neural network training.